# 第5回実習：UR5e＋Robotiq 2F-85による円柱試料の接触把持・持ち上げ（開発途中）

**到達目標**：MuJoCoのUR5e＋Robotiq 2F-85先端に円柱試料（自由物体）を配置し、**左右の指パッドで接触 → 把持保持 → 約75 mmの上昇**が起きるかを調べます。

- **今回実装**：円柱試料と小型支持台／グリッパー閉動作／UR5eの位置・姿勢をほぼ維持した持ち上げ目標姿勢の逆運動学／実際の接触と試料位置のログ／MP4動画。
- **未実装**：外部カメラによる試料位置認識、アームによる試料への接近、置き場所への搬送、力覚フィードバック制御、AIによる把持軌道生成、ROS 2／実機。
- **試料初期位置はプログラムで決めます**。把持位置の自動発見ではありません。
- **物理モデルの摩擦・質量は探索用の暫定値**であり、実物の摩擦係数やRobotiqの性能保証値ではありません。
- **重要：グリッパーを閉じただけでは成功とは判定しません。** 左右パッドの物体接触と試料の高さ・相対位置を独立に評価します。

これまでの01～04ノートブックを**変更しません**。本ファイルを単独でGoogle Colabにアップロードして実行してください。 

> 手順：**セル1 → セル2 → セル3（静止画像を確認）**。問題がなければセル4で軌道計算、セル5で動画・CSV、セル6で結果判定、セル7でダウンロード。

このノートブックは試作コードです。モデルの版や接触パラメータにより失敗し得ます。その場合は失敗ログ自体が次回の検討資料になります。

## 1. セル1：Colab環境準備

右上でCPUランタイムを接続し、以下を実行します。インストール終了後、上から順番に実行してください。 

In [ ]:
# セル1：Colabの依存関係（CPU環境、OSMesa使用）
!apt-get -qq update > /dev/null
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y libosmesa6 libosmesa6-dev fonts-noto-cjk > /dev/null
%pip -q install "mujoco>=3.3,<4" "mujoco-menagerie==2026.10.3" "imageio[ffmpeg]" numpy pandas pillow matplotlib scipy
print('セル1完了。次はセル2を実行してください。')


## 2. セル2：UR5e・2F-85の結合と円柱試料の配置

第3・4回で成功した `MjSpec.attach()` の構成を再使用します。いったん**試料なしモデル**でHOME姿勢の `grip_pinch` の位置を求め、その直下に支持台、その中心に円柱試料を配置したモデルを**新たに組み立て直します**。

寸法と質量は**初期探索条件**です：直径約32 mm、高さ56 mm、質量30 g、支持台直径22 mm。試料は `freejoint` を持ち、グリッパーには溶接されません。 

今回は環境をシンプルにするため、試料を**開いた指の間に初めから置く**方式です。物体への接近軌道は次回以降に検討します。

In [ ]:
# セル2：MuJoCoモデル構築（公式Menagerieモデルを使用）
import os
os.environ['MUJOCO_GL'] = 'osmesa'  # mujoco import より先

import mujoco
import mujoco_menagerie as mm
import numpy as np
import pandas as pd
from pathlib import Path

np.set_printoptions(precision=4, suppress=True)
HOME_Q = np.array([-1.5708, -1.5708, 1.5708, -1.5708, -1.5708, 0.0])
ARM_JOINTS = ['shoulder_pan_joint', 'shoulder_lift_joint', 'elbow_joint',
              'wrist_1_joint', 'wrist_2_joint', 'wrist_3_joint']
ARM_ACTUATORS = ['shoulder_pan', 'shoulder_lift', 'elbow',
                 'wrist_1', 'wrist_2', 'wrist_3']

RADIUS_M = 0.016     # 直径32mm
HALF_HEIGHT_M = 0.028 # 高さ56mm
MASS_KG = 0.030      # 30g：探索用の値
LIFT_M = 0.075       # 75mm
PEDESTAL_HALF_H = 0.035
PEDESTAL_R = 0.011

# 毎回別のMjSpecを作成する。コンパイル済みspecに追加する際のバージョン依存を回避。
def make_spec():
    ur = mm.get('universal_robots_ur5e').spec('scene')
    grip = mm.get('robotiq_2f85').spec('2f85')
    mount = ur.site('attachment_site')
    if mount is None:
        raise RuntimeError('公式UR5eに attachment_site が見つかりません')
    ur.attach(grip, prefix='grip_', site=mount)
    return ur

def get_id(m, kind, name):
    k = mujoco.mj_name2id(m, kind, name)
    if k < 0:
        raise RuntimeError(f'モデル内に {name} がありません')
    return int(k)

def set_home(m, d):
    joint_ids = [get_id(m, mujoco.mjtObj.mjOBJ_JOINT, n) for n in ARM_JOINTS]
    actuator_ids = [get_id(m, mujoco.mjtObj.mjOBJ_ACTUATOR, n) for n in ARM_ACTUATORS]
    mujoco.mj_resetData(m, d)
    for jid, angle in zip(joint_ids, HOME_Q):
        d.qpos[int(m.jnt_qposadr[jid])] = float(angle)
    for aid, angle in zip(actuator_ids, HOME_Q):
        d.ctrl[aid] = float(angle)
    gaid = get_id(m, mujoco.mjtObj.mjOBJ_ACTUATOR, 'grip_fingers_actuator')
    d.ctrl[gaid] = 0.0
    mujoco.mj_forward(m, d)

probe_model = make_spec().compile()
probe_data = mujoco.MjData(probe_model)
set_home(probe_model, probe_data)
probe_pinch_id = get_id(probe_model, mujoco.mjtObj.mjOBJ_SITE, 'grip_pinch')
PINCH_HOME = np.array(probe_data.site_xpos[probe_pinch_id], dtype=float).copy()
PINCH_ROT_HOME = np.array(probe_data.site_xmat[probe_pinch_id]).reshape(3,3).copy()
axis_dot = abs(float(PINCH_ROT_HOME[:,2] @ np.array([0., 0., 1.])))
print('HOMEでの把持点 [m]:', PINCH_HOME, '／把持軸と鉛直軸の平行度:', round(axis_dot,3))
if axis_dot < 0.75:
    raise RuntimeError('HOMEで把持軸が大きく傾いています。円柱の向き・置き方を再検討してください。')
if PINCH_HOME[2] < HALF_HEIGHT_M + 2*PEDESTAL_HALF_H + 0.025:
    raise RuntimeError('試料を置く高さが足りません。HOME姿勢を調整してください。')

# 支持台上面は試料底面よりわずかに低くする（重力で試料が接地する）
SUPPORT_TOP_Z = float(PINCH_HOME[2] - HALF_HEIGHT_M - 0.0005)
PILLAR_CENTER_Z = SUPPORT_TOP_Z - PEDESTAL_HALF_H

spec = make_spec()
spec.worldbody.add_geom(
    name='sample_pedestal', type=mujoco.mjtGeom.mjGEOM_CYLINDER,
    pos=[float(PINCH_HOME[0]), float(PINCH_HOME[1]), PILLAR_CENTER_Z],
    size=[PEDESTAL_R, PEDESTAL_HALF_H, 0.0],
    rgba=[0.23, 0.67, 0.60, 1.0],
    friction=[1.1, 0.005, 0.0001],
)
object_body = spec.worldbody.add_body(name='test_sample', pos=PINCH_HOME.tolist())
object_body.add_freejoint()
object_body.add_geom(
    name='test_sample_cylinder', type=mujoco.mjtGeom.mjGEOM_CYLINDER,
    size=[RADIUS_M, HALF_HEIGHT_M, 0.0],
    mass=MASS_KG, friction=[1.5, 0.01, 0.001],
    rgba=[0.96, 0.56, 0.17, 1.0],
)
model = spec.compile()
data = mujoco.MjData(model)
set_home(model, data)

arm_joint_ids = [get_id(model, mujoco.mjtObj.mjOBJ_JOINT, n) for n in ARM_JOINTS]
arm_qpos_ids = np.array([int(model.jnt_qposadr[j]) for j in arm_joint_ids])
arm_dof_ids = np.array([int(model.jnt_dofadr[j]) for j in arm_joint_ids])
arm_actuator_ids = [get_id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, n) for n in ARM_ACTUATORS]
fr_id = get_id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, 'grip_fingers_actuator')
left_j = get_id(model, mujoco.mjtObj.mjOBJ_JOINT, 'grip_left_driver_joint')
right_j = get_id(model, mujoco.mjtObj.mjOBJ_JOINT, 'grip_right_driver_joint')
left_qpos_id = int(model.jnt_qposadr[left_j]); right_qpos_id = int(model.jnt_qposadr[right_j])
pinch_id = get_id(model, mujoco.mjtObj.mjOBJ_SITE, 'grip_pinch')
obj_geom = get_id(model, mujoco.mjtObj.mjOBJ_GEOM, 'test_sample_cylinder')
obj_body_id = get_id(model, mujoco.mjtObj.mjOBJ_BODY, 'test_sample')
obj_j = model.body_jntadr[obj_body_id]
obj_qpos_id = int(model.jnt_qposadr[obj_j])
left_pads = {get_id(model, mujoco.mjtObj.mjOBJ_GEOM, 'grip_'+n)
             for n in ['left_pad1','left_pad2']}
right_pads = {get_id(model, mujoco.mjtObj.mjOBJ_GEOM, 'grip_'+n)
              for n in ['right_pad1','right_pad2']}

assert model.nu == 7, f'想定外のアクチュエータ数: {model.nu}'
assert model.jnt_type[obj_j] == mujoco.mjtJoint.mjJNT_FREE, '試料は自由物体である必要があります'
assert abs(float(data.qpos[obj_qpos_id+2])-PINCH_HOME[2]) < 0.001
assert np.allclose(data.site_xpos[pinch_id], PINCH_HOME, atol=0.001)
print('モデル構築成功:', 'nq=', model.nq, 'nu=', model.nu)
print('円柱:', f'直径{RADIUS_M*2000:.0f}mm／高さ{HALF_HEIGHT_M*2000:.0f}mm／質量{MASS_KG*1000:.0f}g')
print('支持台上面 z [m]:', round(SUPPORT_TOP_Z,4))
print('グリッパー制御範囲:', model.actuator_ctrlrange[fr_id])
print('次はセル3の静止画像で支持台と試料が重なっていないか確認してください。')


## 3. セル3：接触前の配置を静止画像で確認

**ここで一度停止して確認してください。** オレンジ色の円柱が開いた指の間にあり、その下に緑色の支持台があることを確かめます。試料はグリッパーに固定していません。 

In [ ]:
# セル3：2視点で試料とグリッパーを表示
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Video
import imageio.v2 as imageio

WIDTH, HEIGHT = 480, 360
renderer = mujoco.Renderer(model, height=HEIGHT, width=WIDTH)
whole_cam = mujoco.MjvCamera()
whole_cam.type = mujoco.mjtCamera.mjCAMERA_FREE
whole_cam.azimuth = 135
whole_cam.elevation = -24
whole_cam.distance = 1.7
whole_cam.lookat[:] = [0.,0.,.35]
near_cam = mujoco.MjvCamera()
near_cam.type = mujoco.mjtCamera.mjCAMERA_FREE
near_cam.azimuth = 135
near_cam.elevation = -18
near_cam.distance = 0.36
fontpath = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
font = ImageFont.truetype(fontpath, 18) if Path(fontpath).exists() else ImageFont.load_default()
smallfont = ImageFont.truetype(fontpath, 14) if Path(fontpath).exists() else ImageFont.load_default()

def render_views(phase, cmd, left_touch=False, right_touch=False):
    # ※ mujoco.mj_forward()は物理時間を進めず、現時点の運動学を更新する
    mujoco.mj_forward(model, data)
    renderer.update_scene(data, camera=whole_cam)
    image1 = Image.fromarray(renderer.render().copy())
    near_cam.lookat[:] = data.site_xpos[pinch_id]
    renderer.update_scene(data, camera=near_cam)
    image2 = Image.fromarray(renderer.render().copy())
    canvas = Image.new('RGB', (2*WIDTH, HEIGHT+86), (15,27,42))
    canvas.paste(image1, (0,62)); canvas.paste(image2, (WIDTH,62))
    dr = ImageDraw.Draw(canvas)
    dr.text((12,5),f'UR5e + Robotiq 2F-85 | cylinder | {phase}',font=font,fill='white')
    dr.text((12,36),'全体 (左)',font=smallfont,fill=(182,215,237))
    dr.text((WIDTH+12,36),'指先拡大 (右) | 開閉: '+str(int(cmd))+'/255',font=smallfont,fill=(182,215,237))
    dr.text((12,HEIGHT+68),f'左パッド接触={left_touch}   右パッド接触={right_touch}   ※物体は自由落下可能',
            font=smallfont,fill=(248,213,166))
    return np.array(canvas)

display(Image.fromarray(render_views('INITIAL',0)))
print('確認：オレンジの試料が指の間にあり、緑の台で支えられていますか？')


## 4. セル4：75 mm上昇させる関節目標を数値逆運動学で求める

第4回のような任意の関節角変更ではなく、**現在の把持点位置から鉛直方向に75 mm上昇**する関節目標を、MuJoCoの先端Jacobianを用いた反復計算で求めます。把持点の姿勢変化を抑えるため回転成分も誤差に含めます。

これは幾何学的な目標姿勢計算であり、**障害物回避・接触しない軌道の保証はありません**。上昇先を求められなければ先へ進みません。

In [ ]:
# セル4：UR5eの6軸Jacobianを用いた上昇目標の逆運動学
from scipy.spatial.transform import Rotation

ik_data = mujoco.MjData(model)
set_home(model, ik_data)
start_pos = np.array(ik_data.site_xpos[pinch_id]).copy()
start_rot = np.array(ik_data.site_xmat[pinch_id]).reshape(3,3).copy()
goal_pos = start_pos + np.array([0.,0.,LIFT_M])
q = HOME_Q.copy()
max_iterations = 180
for it in range(max_iterations):
    ik_data.qpos[arm_qpos_ids] = q
    mujoco.mj_forward(model, ik_data)
    now_pos = np.array(ik_data.site_xpos[pinch_id]).copy()
    now_rot = np.array(ik_data.site_xmat[pinch_id]).reshape(3,3).copy()
    pos_err = goal_pos-now_pos
    rot_err = Rotation.from_matrix(start_rot @ now_rot.T).as_rotvec()
    if np.linalg.norm(pos_err) < 0.0025 and np.linalg.norm(rot_err) < 0.09:
        break
    jp = np.zeros((3,model.nv)); jr = np.zeros((3,model.nv))
    mujoco.mj_jacSite(model,ik_data,jp,jr,pinch_id)
    J = np.vstack([jp[:,arm_dof_ids],0.3*jr[:,arm_dof_ids]])
    error = np.r_[pos_err,0.3*rot_err]
    damping = 0.02
    dq = J.T @ np.linalg.solve(J @ J.T + damping**2*np.eye(6), error)
    q += np.clip(dq, -0.09, 0.09)
    # 位置アクチュエータの制御範囲を超えない
    q = np.clip(q,
                np.array([model.actuator_ctrlrange[a,0] for a in arm_actuator_ids]),
                np.array([model.actuator_ctrlrange[a,1] for a in arm_actuator_ids]))

ik_data.qpos[arm_qpos_ids] = q
mujoco.mj_forward(model, ik_data)
actual_pos = np.array(ik_data.site_xpos[pinch_id]).copy()
actual_rot = np.array(ik_data.site_xmat[pinch_id]).reshape(3,3).copy()
p_err = float(np.linalg.norm(actual_pos-goal_pos))
r_err = float(np.linalg.norm(Rotation.from_matrix(start_rot @ actual_rot.T).as_rotvec()))
LIFT_Q = q.copy()
print('HOME関節角 [rad]:',np.round(HOME_Q,3))
print('LIFT関節角 [rad]:',np.round(LIFT_Q,3))
print(f'把持点の目標上昇={LIFT_M*1000:.0f}mm / IK位置誤差={p_err*1000:.1f}mm / 姿勢誤差={r_err:.3f}rad / iterations={it+1}')
if p_err > 0.008 or r_err > 0.12:
    raise RuntimeError('IK目標の精度不足。HOME姿勢またはLIFT_Mを変更して再計算してください。')
print('IK計算成功。次はセル5の物理シミュレーションです。')


## 5. セル5：接触・閉把持・持ち上げを実行し動画とログを作る

次の段階を**物理シミュレーション**で実行します。

1. `SETTLE_OPEN`：指を開いたまま試料を支持台に落ち着かせる。
2. `CLOSE`：指を閉じる（指令255）。
3. `GRASP_HOLD`：閉じた状態を保持。
4. `LIFT`：75 mm上へアームの目標姿勢を滑らかに遷移する。
5. `HOLD_UP`：上昇後も保持し、滑落しないか観察する。

**接触の定義**：円柱のcollision geomと、2F-85の左右パッド（`left_pad1/2`、`right_pad1/2`）間のMuJoCo接触を別々に集計。ロボット全体の `ncon` だけで「把持成功」とは判定しません。

**試料位置は初期化後に強制移動しません。** グリッパー閉動作や摩擦により試料が自然に持ち上がるかを測ります。

In [ ]:
# セル5：接触を識別して物体持ち上げをログ化する
from IPython.display import display, Video

OUT_MP4 = 'ur5e_2f85_grasp_lift_test.mp4'
OUT_CSV = 'ur5e_2f85_grasp_lift_frames.csv'
OUT_SUMMARY = 'ur5e_2f85_grasp_lift_summary.csv'
FPS = 8
# phase, duration_s, arm_goal, gripper_command
PHASES = [
    ('SETTLE_OPEN', 1.25, HOME_Q, 0.0),
    ('CLOSE', 3.0, HOME_Q, 255.0),
    ('GRASP_HOLD', 1.0, HOME_Q, 255.0),
    ('LIFT', 3.0, LIFT_Q, 255.0),
    ('HOLD_UP', 1.5, LIFT_Q, 255.0),
]

def inspect_pad_contacts():
    left_n = 0; right_n = 0
    left_normal_n = 0.; right_normal_n = 0.
    for k in range(data.ncon):
        c = data.contact[k]
        a, b = int(c.geom1), int(c.geom2)
        if a != obj_geom and b != obj_geom:
            continue
        other = b if a == obj_geom else a
        if other in left_pads or other in right_pads:
            cf = np.zeros(6)
            mujoco.mj_contactForce(model, data, k, cf)
            if other in left_pads:
                left_n += 1
                left_normal_n += max(0., float(cf[0]))
            if other in right_pads:
                right_n += 1
                right_normal_n += max(0., float(cf[0]))
    return left_n, right_n, left_normal_n, right_normal_n

# シミュレーションごとに初期状態を再現する
set_home(model,data)
starting_z = float(data.xpos[obj_body_id,2])
frame_count = 0
frame_records = []
phase_records = []
h = float(model.opt.timestep)
if h <= 0:
    raise RuntimeError('MuJoCo時間刻みが無効です')

with imageio.get_writer(OUT_MP4, fps=FPS, codec='libx264', quality=7, macro_block_size=1) as vid:
    for phase, duration, dest_q, gripper_cmd in PHASES:
        phase_start_q = np.asarray([data.ctrl[a] for a in arm_actuator_ids]).copy()
        nframes = int(round(duration*FPS))
        for local_idx in range(nframes):
            frac = (local_idx+1)/nframes
            s = frac*frac*(3.-2.*frac)
            q_command = (1-s)*phase_start_q+s*np.asarray(dest_q)
            for aid, angle in zip(arm_actuator_ids, q_command):
                data.ctrl[aid] = float(angle)
            data.ctrl[fr_id] = float(gripper_cmd)
            desired_time = (frame_count+1)/FPS
            while data.time + h/2 < desired_time:
                mujoco.mj_step(model, data)
            if not np.all(np.isfinite(data.qpos)) or not np.all(np.isfinite(data.qvel)):
                raise RuntimeError(f'{phase}: 数値異常が発生しました。')
            left_n, right_n, left_nf, right_nf = inspect_pad_contacts()
            object_p = np.asarray(data.xpos[obj_body_id]).copy()
            pinch_p = np.asarray(data.site_xpos[pinch_id]).copy()
            clearance = float(object_p[2]-HALF_HEIGHT_M-SUPPORT_TOP_Z)
            frame_records.append({
                'time_s':round(float(data.time),4),
                'phase':phase,
                'gripper_ctrl_0_255':float(data.ctrl[fr_id]),
                'left_driver_rad':float(data.qpos[left_qpos_id]),
                'right_driver_rad':float(data.qpos[right_qpos_id]),
                'left_pad_contact_count':left_n,
                'right_pad_contact_count':right_n,
                'left_pad_normal_force_N':round(left_nf,3),
                'right_pad_normal_force_N':round(right_nf,3),
                'both_pads_touch':bool(left_n>0 and right_n>0),
                'sample_x_m':float(object_p[0]),
                'sample_y_m':float(object_p[1]),
                'sample_z_m':float(object_p[2]),
                'pinch_z_m':float(pinch_p[2]),
                'sample_lift_from_initial_m':float(object_p[2]-starting_z),
                'sample_clearance_above_support_m':clearance,
                'sample_to_pinch_distance_m':float(np.linalg.norm(object_p-pinch_p)),
                'sample_vertical_alignment':abs(float(np.asarray(data.xmat[obj_body_id]).reshape(3,3)[:,2] @ np.array([0.,0.,1.]))),
                'max_arm_tracking_error_rad':float(np.max(np.abs(data.qpos[arm_qpos_ids]-q_command))),
                'contacts_all_n':int(data.ncon),
            })
            vid.append_data(render_views(phase,gripper_cmd,left_n>0,right_n>0))
            frame_count += 1
        last = frame_records[-1]
        phase_records.append({
            'phase':phase, 'end_time_s':last['time_s'],
            'end_sample_z_m':last['sample_z_m'],
            'end_support_clearance_m':last['sample_clearance_above_support_m'],
            'end_both_pads_touch':last['both_pads_touch'],
            'end_arm_tracking_error_rad':last['max_arm_tracking_error_rad'],
        })

df = pd.DataFrame(frame_records)
phase_df = pd.DataFrame(phase_records)
df.to_csv(OUT_CSV,index=False,encoding='utf-8-sig')
phase_df.to_csv(OUT_SUMMARY,index=False,encoding='utf-8-sig')
print('動画:',OUT_MP4,f'({frame_count/FPS:.2f}秒 / {frame_count}フレーム)')
print('フレームCSV:',OUT_CSV, len(df),'行')
print('段階別CSV:',OUT_SUMMARY,len(phase_df),'行')
print(phase_df.to_string(index=False))
display(Video(OUT_MP4,embed=True,width=960))


## 6. セル6：把持・持ち上げ成功の仮判定と可視化

少なくとも、**両側パッドが試料に接触**し、**閉じたまま保持**し、**試料の底面が台から30mm以上離れる**ことを確認します。

以下の結果は設計者が定めた初期の検証基準であり、工業用の把持安全性・把持力保証とは異なります。また、動画の見た目だけで成功と判定しないようにします。

In [ ]:
# セル6：数値による仮判定（合格しなくても研究結果として保存する）
import matplotlib.pyplot as plt

closing = df[df.phase.isin(['CLOSE','GRASP_HOLD'])]
lifting = df[df.phase.isin(['LIFT','HOLD_UP'])]
holding = df[df.phase == 'HOLD_UP']
any_dual_contact = bool(closing.both_pads_touch.any())
continued_contact_rate = float(holding.both_pads_touch.mean()) if len(holding) else 0.
end_clearance = float(df.iloc[-1].sample_clearance_above_support_m)
end_lift = float(df.iloc[-1].sample_lift_from_initial_m)
end_relative_error = float(df.iloc[-1].sample_to_pinch_distance_m)
end_alignment = float(df.iloc[-1].sample_vertical_alignment)

# 合格基準は厳しめの一次スクリーニング。接触が見えても上下保持に失敗したら不合格。
preliminary_success = bool(any_dual_contact and end_clearance > 0.03
                           and end_lift > 0.03 and continued_contact_rate >= 0.5
                           and end_relative_error < 0.045 and end_alignment > 0.94)
print('---- 接触把持と持ち上げの一次評価 ----')
print('CLOSE/HOLDで左右パッド接触あり:',any_dual_contact)
print(f'HOLD_UPで左右接触が記録されたフレーム割合: {continued_contact_rate:.2f}')
print(f'試料の初期位置からの上昇: {end_lift*1000:.1f} mm')
print(f'支持台上面からの試料底面クリアランス: {end_clearance*1000:.1f} mm')
print(f'試料中心と把持点の距離（終了時）: {end_relative_error*1000:.1f} mm')
print(f'終了時の試料長手軸と鉛直の一致度: {end_alignment:.3f}（1.0なら鉛直）')
print('一次判定:', '条件を満たした（追加の再現性試験が必要）' if preliminary_success else '条件未達（要調整・失敗も重要な結果）')

fig,axs=plt.subplots(2,1,figsize=(11,7),sharex=True)
axs[0].plot(df.time_s,df.sample_z_m,label='sample center z')
axs[0].plot(df.time_s,df.pinch_z_m,label='gripper pinch z')
axs[0].axhline(SUPPORT_TOP_Z+HALF_HEIGHT_M,color='gray',linestyle='--',label='sample z on support')
axs[0].set_ylabel('height [m]');axs[0].legend();axs[0].grid(alpha=.25)
axs[1].plot(df.time_s,df.left_pad_normal_force_N,label='left pad normal force [N]')
axs[1].plot(df.time_s,df.right_pad_normal_force_N,label='right pad normal force [N]')
axs[1].set_ylabel('contact normal force [N]');axs[1].set_xlabel('simulated time [s]')
axs[1].legend();axs[1].grid(alpha=.25)
plt.tight_layout();plt.show()
print('※グリッパーのモデルは探索用。接触反力は物理モデルの算出値であり実測値ではありません。')


## 7. セル7：動画とCSVをPCに保存

まず動画と2つのCSVをダウンロードし、このチャットへアップロードしてください。**結果を確認してから**GitHubに05ノートブックを登録します。01～04と既存のYouTube動画は変更しません。

In [ ]:
# セル7：Colabで実行
from google.colab import files
for path in [OUT_MP4,OUT_CSV,OUT_SUMMARY]:
    files.download(path)


## 成功の判断と次の実習

| 項目 | 今回の評価方法 |
|---|---|
| シミュレーションに円柱自由物体を配置 | モデルのfreejointと静止画像 |
| 2F-85の左右パッド接触 | 円柱geomと左右pad geomのMuJoCo接触を別々に検出 |
| 重力に抗した物体の上昇 | 円柱のワールド座標Z、台からのクリアランス |
| 上昇後の保持 | HOLD_UP区間の接触維持率、試料と把持点の距離 |
| 安全性・現実の把持保証 | **今回の対象外** |

**失敗した場合は** `CLOSE` 中の接触、`HOLD_UP` 中の接触保持率、試料位置の変化を順番に確認する。特に試料の直径、位置ずれ、摩擦、質量、モデルのパッド衝突形状、昇降加速度を候補として一つずつ変える。基準を緩くして成功扱いにしない。

**次の第6回実習候補**：試料への接近・把持・鉛直持ち上げ・搬送先への移動・配置（Pick & Place）。第5回の基本把持が再現できた後に進む。

### 参考資料（公式）
- [MuJoCo Python API：MjSpec、Jacobian、レンダリング](https://mujoco.readthedocs.io/en/stable/python.html)
- [MuJoCo Menagerie：Robotiq 2F-85モデル](https://github.com/google-deepmind/mujoco_menagerie/tree/main/robotiq_2f85)
- [MuJoCo Menagerie：UR5eモデル](https://github.com/google-deepmind/mujoco_menagerie/tree/main/universal_robots_ur5e)